In [ ]:
import numpy as np
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

import matplotlib.pyplot as plt

from tqdm import tqdm 

from loka.data import load_breakthrough_states, save_breakthrough_states, load_forgetting_states, save_forgetting_states
from loka.models import BreakthroughCNN
from loka.training_methods import ForgettingScores

import os
import pandas as pd
from dotenv import load_dotenv

SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

load_dotenv()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

BASE = os.getenv("BASE_DIR")

X,Y = load_breakthrough_states(f"{BASE}/data/game_states/e.txt")
print(X.shape)

In [ ]:
x = X[100]

In [ ]:
# https://discuss.pytorch.org/t/k-nearest-neighbor-in-pytorch/59695
# K nearest neighbours :)
x = X[100]
dist = torch.linalg.vector_norm(torch.Tensor(X-x), dim=1)
dist[100] = float('inf') # dont pick myself
knn = dist.topk(20, largest=False)
knn

In [ ]:
# Well really CAL inspired, we are using abs here lol
def CAL_select1(model, X_label, X_pool, size_Q, k_nearest):
    S_xp = torch.zeros(len(X_pool))

    for i in range(len(X_pool)):
        xp = X_pool[i]

        # k-nearest labeled neighbours
        dist = torch.linalg.vector_norm(X_label - xp, dim=1)
        knn = dist.topk(k_nearest, largest=False)

        preds = model(X_label[knn.indices]).squeeze(-1)   # [k]
        pred_xp = model(xp.unsqueeze(0)).squeeze()        # scalar
        S_xp[i] = torch.abs(preds - pred_xp).mean()

    return S_xp.topk(size_Q).indices

In [ ]:
@torch.no_grad()
def CAL_select(model, X_label, X_pool, size_Q, k_nearest, chunk=1024, bs=1024):
    
    @torch.no_grad()
    def predict(model, X, bs=1024):
        out = [model(X[s:s + bs]).reshape(-1) for s in range(0, len(X), bs)]
        return torch.cat(out)
        
    model.eval()
    k = min(k_nearest, len(X_label))

    pred_label = predict(model, X_label, bs)    # [L]
    pred_pool = predict(model, X_pool, bs)      # [P]

    scores = torch.empty(len(X_pool), device=pred_pool.device)
    for s in range(0, len(X_pool), chunk):
        e = s + chunk
        d = torch.cdist(X_pool[s:e], X_label)
        idx = d.topk(k, largest=False).indices
        nb = pred_label[idx]
        scores[s:e] = (nb - pred_pool[s:e, None]).abs().mean(dim=1)

    return scores.topk(size_Q).indices

In [ ]:
X_pool=torch.FloatTensor(X).to(device)
X_label = X_pool[100].unsqueeze(0)

mask = torch.ones(len(X_pool), dtype=torch.bool)
mask[100] = False
X_pool = X_pool[mask]

In [ ]:
model = BreakthroughCNN().to(device)
size_Q = 1000
k_nearest=1

Q_idx = CAL_select(model, X_label, X_pool, size_Q, k_nearest)

# add the selected points to the labeled set
X_label = torch.cat([X_label, X_pool[Q_idx]], dim=0)

# remove them from the pool
mask = torch.ones(len(X_pool), dtype=torch.bool)
mask[Q_idx] = False
X_pool = X_pool[mask]

In [ ]:
X_label.shape